# Phase 5 — Performance Tuning

**Duration:** Weeks 7–8 | **Goal:** Make your Spark jobs 10x faster by understanding what's actually happening.

---

### What We'll Cover

| Area | Key Skills |
| --- | --- |
| Shuffle Tuning | `spark.sql.shuffle.partitions`, repartition vs coalesce, AQE |
| Caching & Persistence | `.cache()`, `.persist()`, when to use (and when NOT to) |
| Broadcast Joins | `broadcast()` hint, auto-broadcast threshold, skew handling |
| Data Skew | Salting, AQE skew join, identifying skew in Spark UI |
| Spark UI | Jobs/Stages/Tasks tabs, reading DAGs, identifying bottlenecks |
| Predicate Pushdown | Filter early, partition pruning, column pruning |
| Photon & Adaptive Query Execution | What they auto-optimize for you |

### Dataset

* **Primary:** `samples.tpch.lineitem` (30M rows) — large enough to show real performance differences
* **Supporting:** `samples.tpch.orders` (7.5M rows) — for join demos

---

**Pre-requisite:** Phases 2–4 completed. You should understand DataFrames, joins, Delta Lake, and file formats.

In [0]:
# ============================================================
# STEP 1: Setup — load datasets for performance experiments
# ============================================================
from pyspark.sql.functions import col, count, sum as spark_sum, avg, lit, spark_partition_id
import time

# Load our datasets
df = spark.read.table("samples.tpch.lineitem")       # 30M rows
df_orders = spark.read.table("samples.tpch.orders")  # 7.5M rows

print(f"lineitem: {df.count():,} rows")
print(f"orders:   {df_orders.count():,} rows")
print(f"\n✅ Ready for performance tuning!")

# Helper function to time operations
def time_it(label, func):
    start = time.time()
    result = func()
    elapsed = time.time() - start
    print(f"  {label}: {elapsed:.2f}s")
    return result, elapsed

## 5.1 — Shuffle Tuning

### What is a shuffle?

A **shuffle** happens whenever Spark needs to reorganize data across executors:
* `groupBy` — all rows with same key must go to same executor
* `join` — matching keys must be on same executor
* `repartition` — explicitly redistributes data
* `orderBy` / `sort` — global sorting requires data movement

### The `spark.sql.shuffle.partitions` setting

**Default: 200.** After any shuffle, data is split into 200 partitions.

| Data Size | Ideal shuffle partitions | Why |
| --- | --- | --- |
| < 100 MB | 1–10 | 200 partitions = 200 tiny tasks (overhead > work) |
| 100 MB – 10 GB | 20–100 | Balanced: enough parallelism, not too much overhead |
| 10–100 GB | 100–500 | Need more tasks for parallel throughput |
| 100 GB+ | 500–2000 | Each task handles ~128 MB |

**Rule of thumb:** Target ~128 MB per partition after shuffle.

### AQE (Adaptive Query Execution) — Spark fixes this for you!

With AQE enabled (default on Databricks), Spark:
* **Coalesces** empty/tiny partitions automatically
* **Splits** oversized partitions (skew handling)
* **Switches** join strategies at runtime based on actual data size

You set 200, but AQE adjusts it down to what's actually needed.

In [0]:
# ============================================================
# STEP 2: Shuffle partitions — impact on performance
# ============================================================
from pyspark.sql.functions import col, count, spark_partition_id
import time

# --- Show default shuffle partitions ---
default_partitions = spark.conf.get("spark.sql.shuffle.partitions")
print(f"Default shuffle partitions: {default_partitions}")

# --- Demo: groupBy with different partition counts ---
print("\nGroupBy (revenue by shipmode) with different shuffle partition settings:")
print("="*60)

# Test with 200 (default)
# NOTE: Even though AQE coalesces partitions on the READ (reduce) side,
# the MAP (write) side still creates all 200 shuffle files.
# That's why 200 partitions is still slower than 2:
#   200 → map writes 200 tiny files → AQE merges on read → ~0.74s
#     2 → map writes 2 files → nothing to merge → ~0.43s
# The extra time = cost of writing 200 small shuffle files + scheduling overhead.
spark.conf.set("spark.sql.shuffle.partitions", "200")
start = time.time()
df.groupBy("l_shipmode").agg(spark_sum(col("l_extendedprice")).alias("revenue")).count()
time_200 = time.time() - start

# Test with 8
spark.conf.set("spark.sql.shuffle.partitions", "8")
start = time.time()
df.groupBy("l_shipmode").agg(spark_sum(col("l_extendedprice")).alias("revenue")).count()
time_8 = time.time() - start

# Test with 2
spark.conf.set("spark.sql.shuffle.partitions", "2")
start = time.time()
df.groupBy("l_shipmode").agg(spark_sum(col("l_extendedprice")).alias("revenue")).count()
time_2 = time.time() - start

# Reset to default
spark.conf.set("spark.sql.shuffle.partitions", "200")

print(f"  200 partitions: {time_200:.2f}s")
print(f"    8 partitions: {time_8:.2f}s")
print(f"    2 partitions: {time_2:.2f}s")
print(f"\n→ AQE auto-coalesces the 200 partitions anyway, so the difference")
print(f"  may be small. But WITHOUT AQE, 200 tiny partitions = 200 tasks = overhead.")

# --- Is AQE enabled? Check the config ---
aqe_enabled = spark.conf.get("spark.sql.adaptive.enabled")
aqe_coalesce = spark.conf.get("spark.sql.adaptive.coalescePartitions.enabled")
aqe_skew = spark.conf.get("spark.sql.adaptive.skewJoin.enabled")
print(f"\nAQE SETTINGS (Adaptive Query Execution):")
print(f"  spark.sql.adaptive.enabled = {aqe_enabled}")
print(f"  spark.sql.adaptive.coalescePartitions.enabled = {aqe_coalesce}")
print(f"  spark.sql.adaptive.skewJoin.enabled = {aqe_skew}")
print(f"  → All TRUE on Databricks by default!")

# Show AQE in action: actual partitions used
spark.conf.set("spark.sql.shuffle.partitions", "200")
df_grouped = df.groupBy("l_shipmode").agg(spark_sum(col("l_extendedprice")).alias("revenue"))
actual_partitions = df_grouped.select(spark_partition_id().alias("pid")).distinct().count()
print(f"\nAQE IN ACTION:")
print(f"  We set shuffle.partitions = 200")
print(f"  But AQE coalesced to: {actual_partitions} partitions")
print(f"  → Only 7 shipmodes exist, so AQE merged 193 empty partitions!")

### Deep Dive: Why Shuffles Are Expensive

**What physically happens during a shuffle:**

1. **Map side:** Each task writes its output to local disk, partitioned by the hash of the shuffle key
2. **Network transfer:** Data is sent across the network to the correct executor
3. **Reduce side:** Each executor reads its partition from all map tasks and merges them

**Cost breakdown:**
* Disk I/O (writing shuffle files)
* Network transfer (moving data between machines)
* Memory pressure (buffering shuffle data)
* Serialization/deserialization

**How to reduce shuffles:**

| Strategy | How |
| --- | --- |
| Filter early | Reduce data BEFORE the groupBy/join (less data to shuffle) |
| Broadcast small tables | Avoids shuffle entirely for joins |
| Use `coalesce` not `repartition` | coalesce = no shuffle (just combines partitions) |
| Pre-partition data on disk | `partitionBy("key")` — data already co-located |
| Avoid unnecessary `orderBy` | Global sort = full shuffle. Do you really need it? |

**AQE auto-optimizations (enabled by default on Databricks):**
* **Coalesce post-shuffle partitions:** Merges tiny partitions into larger ones
* **Skew join optimization:** Splits oversized partitions into smaller pieces
* **Dynamic join selection:** Switches from sort-merge to broadcast if a side is small enough

## 5.2 — Caching & Persistence

### When to cache?

Cache a DataFrame when you **reuse it multiple times** in the same job:
```python
df_expensive = df.join(...).groupBy(...).agg(...)  # expensive to compute
df_expensive.cache()       # store in memory
df_expensive.count()       # triggers materialization

# Now these are fast (read from memory, not recomputed):
df_expensive.filter(...).show()
df_expensive.groupBy(...).agg(...).show()
df_expensive.write.parquet(...)
```

### When NOT to cache:
* DataFrame used only once (caching adds overhead, no benefit)
* DataFrame is too large for memory (spills to disk = slower than recomputing)
* You're reading from Delta (already cached at storage layer with OS page cache)

### `.cache()` vs `.persist()`

| Method | Storage Level | When to use |
| --- | --- | --- |
| `.cache()` | MEMORY_AND_DISK | Default: store in RAM, spill to disk if needed |
| `.persist(MEMORY_ONLY)` | Memory only | Fast but fails if data doesn’t fit |
| `.persist(DISK_ONLY)` | Disk only | Large data you reuse but can’t fit in RAM |
| `.persist(MEMORY_AND_DISK_SER)` | Serialized | Saves memory space (compressed) but slower to read |

### Always `.unpersist()` when done!
```python
df_expensive.unpersist()  # free the memory
```

In [0]:
# ============================================================
# STEP 3: Caching — when it helps and when it doesn't
# ============================================================
import time
from pyspark.sql.functions import col, sum as spark_sum, avg

# --- Create an expensive transformation ---
df_expensive = df.join(df_orders, df["l_orderkey"] == df_orders["o_orderkey"]) \
    .select(
        df["l_orderkey"], "l_shipmode", "l_extendedprice", "l_discount",
        "o_orderdate", "o_orderstatus", "o_totalprice"
    )

# --- WITHOUT cache: re-computed each time ---
print("WITHOUT CACHE (recomputes the join each time):")
start = time.time()
df_expensive.groupBy("l_shipmode").agg(spark_sum("l_extendedprice").alias("rev")).count()
first_no_cache = time.time() - start

start = time.time()
df_expensive.groupBy("o_orderstatus").agg(avg("o_totalprice").alias("avg_price")).count()
second_no_cache = time.time() - start

print(f"  Query 1 (groupBy shipmode): {first_no_cache:.2f}s")
print(f"  Query 2 (groupBy status):   {second_no_cache:.2f}s")
print(f"  Total: {first_no_cache + second_no_cache:.2f}s")

# --- WITH cache: computed once, reused from memory ---
print("\nWITH CACHE (join computed once, reused):")
df_expensive.cache()

# First query triggers materialization (fills the cache)
start = time.time()
df_expensive.groupBy("l_shipmode").agg(spark_sum("l_extendedprice").alias("rev")).count()
first_cached = time.time() - start

# Second query reads from cache (no recomputation!)
start = time.time()
df_expensive.groupBy("o_orderstatus").agg(avg("o_totalprice").alias("avg_price")).count()
second_cached = time.time() - start

print(f"  Query 1 (materialize + compute): {first_cached:.2f}s")
print(f"  Query 2 (from cache!):           {second_cached:.2f}s")
print(f"  Total: {first_cached + second_cached:.2f}s")

# Cleanup
df_expensive.unpersist()
print("\n✅ Cache released with .unpersist()")

### Deep Dive: Caching Pitfalls

**The #1 mistake:** Caching everything “just in case.”

Caching uses executor memory. If you cache too much:
* Spark runs out of memory for actual computation
* Cached data spills to disk (slower than recomputing from Parquet!)
* GC pressure increases, slowing everything down

**Decision framework:**

```
Will I reuse this DataFrame more than once?
  NO  → Don't cache
  YES → Is it expensive to recompute (join, groupBy, complex logic)?
    NO  → Don't cache (just re-read from Delta/Parquet)
    YES → Does it fit in memory?
      NO  → Consider persist(DISK_ONLY) or just recompute
      YES → Cache it!
```

**Why Delta/Parquet often makes caching unnecessary:**

Delta files on cloud storage benefit from:
* OS page cache (recently-read files stay in RAM automatically)
* Databricks IO cache (SSD-backed, transparent)
* Photon vectorized reading (extremely fast even without explicit cache)

So `spark.read.table("my_delta_table")` twice is often nearly as fast as `.cache()` — without using executor memory.

**When caching IS essential:**
* Iterative ML algorithms (reading the same training data 100+ times)
* Interactive exploration (running many ad-hoc queries on the same transformed DF)
* Streaming micro-batches that reuse state

## 5.3 — Broadcast Joins & Join Strategies

### The Problem with Default Joins

Default join (Sort-Merge Join):
1. Shuffle BOTH sides by the join key
2. Sort both sides
3. Merge matching keys

Cost: 2 full shuffles + 2 sorts. For two 30M-row tables = very expensive.

### Broadcast Join: Skip the shuffle entirely

If one side is small (< 10 MB by default), Spark **copies it to every executor**. The big side never moves.

```python
from pyspark.sql.functions import broadcast

# Force broadcast of small table
df_big.join(broadcast(df_small), on="key")
```

### Join Strategies (from fastest to slowest):

| Strategy | When | Cost |
| --- | --- | --- |
| **Broadcast Hash Join** | One side < 10 MB (or forced with `broadcast()`) | No shuffle! Fastest. |
| **Shuffle Hash Join** | Medium tables, one side much smaller | 1 shuffle |
| **Sort Merge Join** | Both sides large, equi-join | 2 shuffles + 2 sorts (default) |
| **Broadcast Nested Loop** | Non-equi joins (>, <, BETWEEN) + one small side | No shuffle but O(n×m) |
| **Cartesian Product** | No join condition / cross join | Extremely expensive |

### Auto-broadcast threshold
```python
spark.conf.get("spark.sql.autoBroadcastJoinThreshold")  # default: 10485760 (10 MB)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "50m")  # increase to 50 MB
```

In [0]:
# ============================================================
# STEP 4: Join strategies — broadcast vs sort-merge
# ============================================================
import time
from pyspark.sql.functions import col

# Load tables
df_nations = spark.read.table("samples.tpch.nation")     # 25 rows!
df_suppliers = spark.read.table("samples.tpch.supplier") # 50K rows

print(f"Sizes: lineitem={30_000_000:,}, orders={7_500_000:,}, nations={25}, suppliers={50_000:,}")
print("="*60)

# --- Sort-Merge Join (default for two large tables) ---
print("\n1. SORT-MERGE JOIN (lineitem × orders — both large):")
start = time.time()
df_smj = df.join(df_orders, df["l_orderkey"] == df_orders["o_orderkey"])
df_smj.select("l_orderkey", "l_shipmode", "o_orderstatus").count()
smj_time = time.time() - start
print(f"   Time: {smj_time:.2f}s (2 shuffles + 2 sorts)")

# --- Broadcast Join using SQL hint (Spark Connect compatible) ---
print("\n2. BROADCAST JOIN (suppliers × nations — nations is tiny):")
# On Spark Connect, use SQL broadcast hint instead of broadcast() function
df_nations.createOrReplaceTempView("nations")
df_suppliers.createOrReplaceTempView("suppliers")
start = time.time()
df_bj = spark.sql("""
    SELECT /*+ BROADCAST(nations) */ s.*, n.n_name
    FROM suppliers s JOIN nations n ON s.s_nationkey = n.n_nationkey
""")
df_bj.count()
bj_time = time.time() - start
print(f"   Time: {bj_time:.2f}s (no shuffle — nations sent to all executors)")

# --- Broadcast hint on medium table ---
print("\n3. BROADCAST HINT on orders (force broadcast into lineitem join):")
df.createOrReplaceTempView("lineitem")
df_orders.createOrReplaceTempView("orders")
start = time.time()
df_forced = spark.sql("""
    SELECT /*+ BROADCAST(orders) */ l.l_orderkey, l.l_shipmode, o.o_orderstatus
    FROM lineitem l JOIN orders o ON l.l_orderkey = o.o_orderkey
""")
df_forced.count()
forced_time = time.time() - start
print(f"   Time: {forced_time:.2f}s (orders broadcast — risky if too large!)")

print(f"\n→ Sort-Merge: {smj_time:.2f}s vs Broadcast hint: {forced_time:.2f}s")
if forced_time < smj_time:
    print("  Broadcast won! But only works if the table fits in executor memory.")
else:
    print("  Sort-Merge was faster here — broadcasting 7.5M rows has overhead.")
print("\n  Note: On Spark Connect, use SQL hints /*+ BROADCAST(table) */")
print("  instead of the broadcast() function.")

### Deep Dive: When Broadcast Joins Go Wrong

**The danger:** If you broadcast a table that’s TOO large, it must fit in **every executor’s memory**. Broadcasting a 2 GB table to 10 executors = 20 GB total memory used!

**Symptoms of a bad broadcast:**
* `OutOfMemoryError` on executors
* Job hangs during broadcast exchange
* Spark UI shows “BroadcastExchange” taking minutes

**Rule of thumb:**
* < 10 MB: Auto-broadcast (no hint needed)
* 10 MB – 200 MB: Consider `broadcast()` hint if you KNOW it fits
* 200 MB – 1 GB: Risky — test first
* > 1 GB: Never broadcast. Use Sort-Merge Join.

**How to check join strategy in the plan:**
```python
df_joined.explain(True)  # Look for:
# - BroadcastHashJoin → broadcast was used
# - SortMergeJoin → both sides shuffled
# - BroadcastNestedLoopJoin → non-equi join
```

**AQE dynamic broadcast:** Even if both tables look large initially, AQE checks runtime sizes. If one side turns out smaller than expected after filtering, AQE may switch to broadcast at runtime!

## 5.4 — Data Skew: The Silent Performance Killer

### What is skew?

Skew = one partition has WAY more data than others.

```
Partition 1:  1,000 rows     ← done in 0.1s
Partition 2:  1,000 rows     ← done in 0.1s
Partition 3:  10,000,000 rows ← done in 60s   ← THE STRAGGLER
```

Your job takes 60s because **you’re only as fast as your slowest task.**

### Common causes:
* **Hot keys:** 90% of orders belong to one customer
* **Null keys:** Millions of nulls grouped into one partition
* **Popular values:** `country = 'US'` has 100x more rows than `country = 'Luxembourg'`

### How to detect skew:
1. **Spark UI → Stages → Task metrics:** One task takes 100x longer than median
2. **`groupBy(key).count().orderBy(desc("count"))`:** Check value distribution
3. **Spark UI → SQL tab:** Look for “data size” imbalance across partitions

### Solutions:

| Solution | How it works |
| --- | --- |
| **AQE Skew Join** (automatic) | Spark detects oversized partitions and splits them at runtime |
| **Salting** (manual) | Add random suffix to hot key → spreads across more partitions |
| **Filtering hot keys** | Process hot keys separately with broadcast join |
| **Increase partitions** | More partitions = smaller chunks (helps mild skew) |

In [0]:
# ============================================================
# STEP 5: Data skew — detect and fix with salting
# ============================================================
from pyspark.sql.functions import (
    col, count, floor, rand, concat, lit, sum as spark_sum, desc, when
)
import time

# --- Create a skewed dataset ---
# 90% of data has l_returnflag = 'N', only 10% has 'R' or 'A'
print("DATA DISTRIBUTION (natural skew in lineitem):")
df.groupBy("l_returnflag").agg(
    count("*").alias("row_count"),
).orderBy(desc("row_count")).show()

# --- Create EXTREME skew for demo ---
print("\nCreating EXTREME skew (1 hot key = 95% of data):")
df_skewed = df.select("l_orderkey", "l_extendedprice", "l_shipmode").withColumn(
    "skewed_key",
    when(rand() < 0.95, lit("HOT_KEY")).otherwise(col("l_shipmode"))
)

df_skewed.groupBy("skewed_key").count().orderBy(desc("count")).show()

# --- Time a groupBy on the skewed key ---
print("GroupBy on SKEWED key (1 partition does 95% of work):")
start = time.time()
df_skewed.groupBy("skewed_key").agg(
    spark_sum("l_extendedprice").alias("revenue")
).count()
skewed_time = time.time() - start
print(f"  Time: {skewed_time:.2f}s")

# --- FIX WITH SALTING: spread the hot key across multiple partitions ---
print("\nFIX: Salting the hot key (spread across 10 sub-partitions):")
NUM_SALTS = 10

# Add a random salt (0-9) to the key
df_salted = df_skewed.withColumn(
    "salted_key",
    concat(col("skewed_key"), lit("_"), floor(rand() * NUM_SALTS).cast("int"))
)

# GroupBy salted key, then aggregate again by original key
start = time.time()
df_partial = df_salted.groupBy("salted_key", "skewed_key").agg(
    spark_sum("l_extendedprice").alias("partial_revenue")
)
df_final = df_partial.groupBy("skewed_key").agg(
    spark_sum("partial_revenue").alias("revenue")
)
df_final.count()
salted_time = time.time() - start
print(f"  Time: {salted_time:.2f}s")

print(f"\n→ Skewed: {skewed_time:.2f}s vs Salted: {salted_time:.2f}s")
print("  Salting spreads HOT_KEY across 10 tasks instead of 1!")
print("  Note: AQE also handles this automatically for joins (skew join optimization).")

### Deep Dive: Salting Explained Step by Step

**The problem:** `groupBy("customer_id")` where customer #1 has 10M orders, everyone else has ~100.

Spark hashes `customer_id` → partition 47 gets 10M rows. Partition 47’s task runs for 10 minutes while all other tasks finish in seconds.

**The salting fix (2-phase aggregation):**

**Phase 1:** Add random salt and do partial aggregation:
```
customer_1_salt_0  → sum(revenue) of ~1M rows
customer_1_salt_1  → sum(revenue) of ~1M rows
...
customer_1_salt_9  → sum(revenue) of ~1M rows
```
10 tasks share the work (1M rows each instead of 10M in one task).

**Phase 2:** Strip the salt, aggregate the partial results:
```
customer_1 → sum of all 10 partial sums
```
This final step is tiny (10 rows to aggregate).

**Trade-off:** Salting adds an extra shuffle (2 shuffles instead of 1). Worth it ONLY when skew is severe.

**When to use salting vs let AQE handle it:**
* **AQE skew join** — handles skew in JOINs automatically. Enable with:
  `spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")` (default: true)
* **Manual salting** — needed for skewed `groupBy` (AQE doesn’t auto-salt aggregations yet)

**Detecting skew in Spark UI:**
1. Go to Stages tab
2. Look at Task Duration metrics
3. If max task time >> median task time (e.g., 60s vs 0.5s) → you have skew
4. Click the stage → check “Shuffle Read Size” per task to find the hot partition

## 5.5 — Predicate Pushdown & Column Pruning

### Predicate Pushdown = “Filter at the source”

When you write `.filter(col("date") == "2024-01-15")`, Spark pushes that filter down to the FILE READER. Files that can’t contain that date are never read.

**How it works with Delta/Parquet:**
1. Each Parquet file footer has min/max stats per column per row group
2. Delta’s transaction log stores file-level min/max stats
3. If `date = 2024-01-15` is outside a file’s [min, max] range → skip the file entirely

### Column Pruning = “Read only needed columns”

Parquet is columnar — if you select 3 of 50 columns, only those 3 are read from disk.

```python
# BAD: reads all 16 columns from disk, then filters
df.filter(col("l_shipmode") == "AIR").select("l_orderkey", "l_extendedprice")

# GOOD: Spark automatically pushes both down
# (reads only 3 columns, skips non-AIR files)
# Same result, same code — Catalyst optimizes the plan!
```

### The “Filter Early” Rule

Always filter/select BEFORE expensive operations (joins, groupBy):
```python
# ✅ Good: filter 30M → 5M rows BEFORE joining
df.filter(col("l_shipdate") > "1995-01-01").join(df_orders, ...)

# ❌ Bad: join 30M rows, THEN filter (shuffled 30M unnecessarily)
df.join(df_orders, ...).filter(col("l_shipdate") > "1995-01-01")
```
Note: Catalyst often reorders these for you, but don’t rely on it for complex queries.

In [0]:
# ============================================================
# STEP 6: Predicate pushdown & column pruning in action
# ============================================================
import time
from pyspark.sql.functions import col

print("PREDICATE PUSHDOWN: Filter early vs filter late")
print("="*60)

# --- Scan everything, then filter ---
print("\n1. Full scan (no filter):")
start = time.time()
df.select("l_orderkey", "l_extendedprice", "l_shipmode").count()
full_scan = time.time() - start
print(f"   Time: {full_scan:.2f}s (read all 30M rows)")

# --- Filter pushdown (only read matching data) ---
print("\n2. With filter pushdown (l_shipdate = '1995-06-15'):")
start = time.time()
df.filter(col("l_shipdate") == "1995-06-15") \
  .select("l_orderkey", "l_extendedprice", "l_shipmode").count()
filtered_scan = time.time() - start
print(f"   Time: {filtered_scan:.2f}s (skipped files not containing that date!)")

# --- Show the physical plan (see PushedFilters) ---
print("\n3. Physical Plan (look for 'PushedFilters'):")
df.filter(col("l_shipdate") == "1995-06-15") \
  .select("l_orderkey", "l_extendedprice") \
  .explain(True)

# --- Column pruning effect ---
print("\n4. COLUMN PRUNING: Reading 2 vs 16 columns")
start = time.time()
df.select("l_orderkey", "l_extendedprice").count()
two_cols = time.time() - start

start = time.time()
df.select("*").count()
all_cols = time.time() - start

print(f"   2 columns:  {two_cols:.2f}s")
print(f"   16 columns: {all_cols:.2f}s")
print("   → Columnar format means less columns = less I/O!")

### Deep Dive: Reading Explain Plans

When you run `df.explain(True)`, you get 4 plan levels:

| Plan | What it shows |
| --- | --- |
| **Parsed Logical** | Your code as Spark understood it |
| **Analyzed Logical** | Column types resolved, tables found |
| **Optimized Logical** | Catalyst’s optimized version (filters pushed down, columns pruned) |
| **Physical Plan** | The actual execution strategy (which joins, which scans) |

**What to look for in the Physical Plan:**

| Keyword | Meaning |
| --- | --- |
| `PhotonBroadcastHashJoin` | Broadcast join (fast!) |
| `PhotonSortMergeJoin` | Sort-merge join (shuffles both sides) |
| `PhotonFilter` / `PushedFilters` | Filters applied at scan level |
| `ReadSchema` | Which columns are actually read (column pruning) |
| `PartitionFilters` | Partition pruning (entire folders skipped) |
| `Exchange` | A shuffle is happening here |
| `AQEShuffleRead` | AQE optimized the shuffle at runtime |

**Quick health check — look for:**
* ✅ `PushedFilters` not empty (filters pushed to storage)
* ✅ `ReadSchema` has only needed columns (not all 16)
* ✅ `BroadcastHashJoin` for small table joins
* ❌ `Exchange` before a filter (you filtered TOO LATE)
* ❌ `CartesianProduct` (missing join condition!)

## 5.6 — Reading the Spark UI

### Where to find it

In Databricks: Click your cluster → **Spark UI** tab. Or in a notebook: **View → Spark UI** link after running a cell.

### The 4 tabs you need:

| Tab | What it shows | What to look for |
| --- | --- | --- |
| **Jobs** | One job per action (count, show, write) | Total duration, failed jobs |
| **Stages** | One stage per shuffle boundary | Input/output size, shuffle read/write |
| **Tasks** | Individual tasks within a stage | Duration distribution (skew = one task way slower) |
| **SQL** | Query plan visualization | DAG shape, data flow, exchange nodes |

### Diagnosing common problems:

| Symptom in Spark UI | Likely cause | Fix |
| --- | --- | --- |
| One task takes 100x longer than median | Data skew | Salt the key or use AQE skew join |
| Many tasks, each processing tiny data | Too many shuffle partitions | Reduce partitions or let AQE coalesce |
| Shuffle write/read is huge | Unnecessary shuffle | Filter earlier, use broadcast, avoid repartition |
| Spill to disk (memory → disk) | Not enough memory per task | Increase partitions (smaller chunks) or add executors |
| “BroadcastExchange” takes minutes | Broadcasting a table that’s too large | Remove broadcast hint, use sort-merge |
| GC time > 10% of task time | Memory pressure | Reduce cache, increase executor memory |

In [0]:
# ============================================================
# STEP 7: Putting it all together — optimizing a real query
# ============================================================
import time
from pyspark.sql.functions import col, sum as spark_sum, avg, year, desc

print("SCENARIO: 'Revenue by nation for 1995 AIR shipments'")
print("="*60)

df_nations = spark.read.table("samples.tpch.nation")
df_suppliers = spark.read.table("samples.tpch.supplier")

# --- UNOPTIMIZED: joins first, filter last ---
print("\n1. UNOPTIMIZED (filter late, joins everything first):")
start = time.time()
result_slow = df \
    .join(df_orders, df["l_orderkey"] == df_orders["o_orderkey"]) \
    .join(df_suppliers, df["l_suppkey"] == df_suppliers["s_suppkey"]) \
    .join(df_nations, df_suppliers["s_nationkey"] == df_nations["n_nationkey"]) \
    .filter((col("l_shipmode") == "AIR") & (year(col("l_shipdate")) == 1995)) \
    .groupBy("n_name") \
    .agg(spark_sum(col("l_extendedprice") * (1 - col("l_discount"))).alias("revenue")) \
    .orderBy(desc("revenue"))
result_slow.count()
slow_time = time.time() - start
print(f"   Time: {slow_time:.2f}s")

# --- OPTIMIZED: filter first, select only needed columns, use SQL broadcast hints ---
print("\n2. OPTIMIZED (filter early, column pruning, broadcast hints via SQL):")
start = time.time()

# Filter EARLY — reduce 30M rows to only AIR + 1995 BEFORE joining
df_filtered = df.filter(
    (col("l_shipmode") == "AIR") & (year(col("l_shipdate")) == 1995)
).select("l_orderkey", "l_suppkey", "l_extendedprice", "l_discount")

# Register filtered data as temp view for SQL broadcast hints
df_filtered.createOrReplaceTempView("lineitem_filtered")
df_orders.select("o_orderkey").createOrReplaceTempView("orders_keys")
df_suppliers.select("s_suppkey", "s_nationkey").createOrReplaceTempView("suppliers_slim")
df_nations.createOrReplaceTempView("nations")

result_fast = spark.sql("""
    SELECT /*+ BROADCAST(suppliers_slim), BROADCAST(nations) */
        n.n_name,
        SUM(lf.l_extendedprice * (1 - lf.l_discount)) AS revenue
    FROM lineitem_filtered lf
    JOIN orders_keys o ON lf.l_orderkey = o.o_orderkey
    JOIN suppliers_slim s ON lf.l_suppkey = s.s_suppkey
    JOIN nations n ON s.s_nationkey = n.n_nationkey
    GROUP BY n.n_name
    ORDER BY revenue DESC
""")
result_fast.count()
fast_time = time.time() - start
print(f"   Time: {fast_time:.2f}s")

speedup = slow_time / fast_time if fast_time > 0 else 0
print(f"\n→ Speedup: {speedup:.1f}x faster!")
print("  Optimizations applied:")
print("  1. Filter early (30M → ~1.7M rows before any join)")
print("  2. Select only needed columns (column pruning)")
print("  3. Broadcast small tables via SQL hint (nations=25, suppliers=50K)")

print("\nFinal result (top 5 nations by AIR revenue in 1995):")
result_fast.show(5)

## Phase 5 Checkpoint

**Can you do these?**

- [ ] Explain what a shuffle is and why it’s expensive
- [ ] Set `spark.sql.shuffle.partitions` and explain AQE coalescing
- [ ] Use `.cache()` correctly and know when NOT to cache
- [ ] Force a broadcast join and explain when it’s dangerous
- [ ] Detect data skew using `groupBy().count()` and Spark UI task durations
- [ ] Fix skew with salting (2-phase aggregation)
- [ ] Read an `explain(True)` plan and identify pushed filters, join types, exchanges
- [ ] Apply the “filter early, select only needed columns, broadcast small tables” pattern

---

**Key takeaways:**
1. **Filter early, select less** — reduce data volume before expensive operations
2. **Broadcast small tables** — eliminates shuffle for joins with small sides
3. **AQE handles most tuning** — coalesces partitions, handles skew, switches join strategies
4. **Cache only expensive + reused DataFrames** — don’t cache everything
5. **Skew = one slow task** — detect via Spark UI, fix with salting or AQE
6. **Read the plan** — `explain(True)` tells you exactly what Spark will do

---

**Next up:** Phase 6 — Structured Streaming (triggers, watermarks, checkpoints, exactly-once)